### 1. IMPORTS & SETUP

In [ ]:
import pandas as pd
import json
import os
from datetime import datetime
import yfinance as yf
import random
import time
import re
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ All imports successful!")
print(f"📅 Current date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check API key
api_key = os.getenv('OPENAI_API_KEY')
print(f"🔑 OpenAI API Key: {'✅ Found' if api_key else '❌ Missing - set OPENAI_API_KEY in .env file'}")

✅ All imports successful!
📅 Current date: 2025-07-28 11:52:29
🔑 OpenAI API Key: ✅ Found


### 2. DEFINE STOCK SYMBOLS TO ANALYZE

In [2]:
# Start with some well-known tech stocks for testing
test_symbols = ["AAPL", "MSFT", "GOOGL", "TSLA", "NVDA", "META", "AMZN", "NFLX"]

print("📊 Stocks we'll analyze:")
for symbol in test_symbols:
    print(f"   - {symbol}")

print(f"\nTotal symbols: {len(test_symbols)}")

📊 Stocks we'll analyze:
   - AAPL
   - MSFT
   - GOOGL
   - TSLA
   - NVDA
   - META
   - AMZN
   - NFLX

Total symbols: 8


### 3. FETCH REAL-TIME MARKET DATA

In [ ]:
print("📡 Fetching real-time market data from Yahoo Finance...")
market_data = {}

for symbol in test_symbols:
    try:
        print(f"   Fetching {symbol}...")
        
        #yf.Ticker is a class provided by the yfinance Python library, 
        # which allows you to retrieve real-time and historical financial data 
        # for a given stock or financial instrument

        # Create ticker object
        ticker = yf.Ticker(symbol) 
          
        # Get different time periods
        hist_1m = ticker.history(period="1mo")  # 1 month data
        hist_1y = ticker.history(period="1y")   # 1 year data
        info = ticker.info                      # Company info
        
        if not hist_1m.empty:
            # Current price and 1-month data
            current_price = hist_1m['Close'].iloc[-1]
            start_1m = hist_1m['Close'].iloc[0]
            high_1m = hist_1m['High'].max()
            low_1m = hist_1m['Low'].min()
            
            # 1-year data
            high_52w = hist_1y['High'].max() if not hist_1y.empty else high_1m
            low_52w = hist_1y['Low'].min() if not hist_1y.empty else low_1m
            
            # Calculate 1-month performance
            performance_1m = ((current_price - start_1m) / start_1m) * 100
            
            # Store all the data
            market_data[symbol] = {
                "current_price": round(float(current_price), 2),
                "1m_high": round(float(high_1m), 2),
                "1m_low": round(float(low_1m), 2),
                "1m_performance": round(float(performance_1m), 2),
                "dividend_yield": round(info.get("dividendYield", 0) * 100, 2) if info.get("dividendYield") else 0,
                "sector": info.get("sector", "Unknown"),
                "market_cap": info.get("marketCap", 0),
                "52w_high": round(float(high_52w), 2),
                "52w_low": round(float(low_52w), 2),
                "company_name": info.get("longName", symbol)
            }
            
            print(f"      ✅ ${current_price:.2f}, 1M: {performance_1m:+.1f}%, Sector: {info.get('sector', 'Unknown')}")
        else:
            print(f"      ❌ No data available")
            
    except Exception as e:
        print(f"      ❌ Error: {e}")
        continue

print(f"\n📊 Successfully fetched data for {len(market_data)} stocks")

📡 Fetching real-time market data from Yahoo Finance...
   Fetching AAPL...
      ✅ $213.82, 1M: +4.2%, Sector: Technology
   Fetching MSFT...
      ✅ $511.71, 1M: +2.9%, Sector: Technology
   Fetching GOOGL...
      ✅ $192.06, 1M: +9.0%, Sector: Communication Services
   Fetching TSLA...
      ✅ $327.25, 1M: +3.0%, Sector: Consumer Cyclical
   Fetching NVDA...
      ✅ $175.58, 1M: +11.1%, Sector: Technology
   Fetching META...
      ✅ $714.40, 1M: -3.2%, Sector: Communication Services
   Fetching AMZN...
      ✅ $232.92, 1M: +6.2%, Sector: Consumer Cyclical
   Fetching NFLX...
      ✅ $1175.46, 1M: -12.2%, Sector: Communication Services

📊 Successfully fetched data for 8 stocks


### 4. DISPLAY FETCHED MARKET DATA

In [4]:
print("📈 REAL-TIME MARKET DATA SUMMARY:")
print("=" * 80)

for symbol, data in market_data.items():
    print(f"{symbol:6} | ${data['current_price']:8.2f} | {data['1m_performance']:+6.1f}% | {data['sector']:15} | {data['company_name']}")

print("=" * 80)
print(f"Total stocks with data: {len(market_data)}")

📈 REAL-TIME MARKET DATA SUMMARY:
AAPL   | $  213.82 |   +4.2% | Technology      | Apple Inc.
MSFT   | $  511.71 |   +2.9% | Technology      | Microsoft Corporation
GOOGL  | $  192.06 |   +9.0% | Communication Services | Alphabet Inc.
TSLA   | $  327.25 |   +3.0% | Consumer Cyclical | Tesla, Inc.
NVDA   | $  175.58 |  +11.1% | Technology      | NVIDIA Corporation
META   | $  714.40 |   -3.2% | Communication Services | Meta Platforms, Inc.
AMZN   | $  232.92 |   +6.2% | Consumer Cyclical | Amazon.com, Inc.
NFLX   | $ 1175.46 |  -12.2% | Communication Services | Netflix, Inc.
Total stocks with data: 8


### 5. SETUP AI DISCOVERY (OpenAI API)

In [5]:
# AI configuration
if api_key:
    import openai
    client = openai.OpenAI(api_key=api_key)
    print("✅ OpenAI client initialized")
else:
    print("❌ No OpenAI API key - will use sample data for demo")

# Generate session variables for varied AI responses
session_id = datetime.now().strftime("%Y%m%d_%H%M%S")
current_date = datetime.now().strftime("%B %Y")
time_seed = int(time.time()) % 1000

print(f"🤖 Session ID: {session_id}")
print(f"📅 Current date for AI: {current_date}")
print(f"🎲 Time seed: {time_seed}")

✅ OpenAI client initialized
🤖 Session ID: 20250728_115929
📅 Current date for AI: July 2025
🎲 Time seed: 569


### 6. AI DISCOVERY PROMPT CREATION

In [6]:
# Choose a sector to analyze
sector = "Technology"
print(f"🔍 AI will discover stocks in: {sector}")

# Create varied market contexts for different results each time
market_contexts = [
    "focus on emerging companies with recent catalysts",
    "include a mix of established and growth companies", 
    "emphasize companies with strong recent earnings",
    "highlight companies with innovation and disruption potential",
    "focus on companies with strong technical momentum"
]

# Randomly select context for this session
market_context = random.choice(market_contexts)
temperature = random.uniform(0.3, 0.7)  # Randomize AI creativity

print(f"🎯 Selected context: {market_context}")
print(f"🌡️ AI temperature: {temperature:.2f}")

🔍 AI will discover stocks in: Technology
🎯 Selected context: include a mix of established and growth companies
🌡️ AI temperature: 0.51


### 7. CREATE AI PROMPT

In [7]:
ai_prompt = f"""Find 8-12 trending stocks in the {sector} sector for {current_date}.

Research Context: {market_context}
Session: {time_seed}

REQUIREMENTS:
- Focus on REAL publicly traded companies only
- Include varied market caps: large-cap, mid-cap, small-cap
- Avoid always suggesting the same popular stocks
- Include companies with recent news, earnings, or catalysts
- Mix established leaders with emerging growth companies

Provide information in this EXACT format:

**STOCK 1:**
Symbol: [TICKER]
Company: [Full Company Name]
Founded: [Year]
Performance: [Expected % return number only]
Category: [large-cap/mid-cap/small-cap]
Catalyst: [Recent catalyst or driver]
Thesis: [Investment reasoning]

**STOCK 2:**
Symbol: [TICKER]
Company: [Full Company Name]
Founded: [Year]
Performance: [Expected % return number only]
Category: [large-cap/mid-cap/small-cap]
Catalyst: [Recent catalyst or driver]
Thesis: [Investment reasoning]

[Continue for 8-12 stocks...]

Focus on providing VARIED and CURRENT {sector} recommendations."""

print("📝 AI Prompt created:")
print(f"   Length: {len(ai_prompt)} characters")
print(f"   Sector: {sector}")
print(f"   Context: {market_context}")

📝 AI Prompt created:
   Length: 1038 characters
   Sector: Technology
   Context: include a mix of established and growth companies


### 8. CALL OPENAI API

In [8]:

if api_key:
    print("🤖 Calling OpenAI API...")
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a financial research expert. Provide accurate, varied, and current recommendations. Avoid giving the same recommendations repeatedly."},
                {"role": "user", "content": ai_prompt}
            ],
            max_tokens=2000,
            temperature=temperature
        )
        
        ai_response = response.choices[0].message.content
        print(f"✅ AI response received!")
        print(f"   Response length: {len(ai_response)} characters")
        print(f"   Model used: {response.model}")
        
        # Show first 300 characters of response
        print(f"\n📝 Response preview:")
        print(ai_response[:300] + "..." if len(ai_response) > 300 else ai_response)
        
    except Exception as e:
        print(f"❌ OpenAI API error: {e}")
        ai_response = None
        
else:
    print("⚠️ Using sample AI response for demo...")
    ai_response = """**STOCK 1:**
Symbol: NVDA
Company: NVIDIA Corporation
Founded: 1993
Performance: 15.2
Category: large-cap
Catalyst: AI chip demand surge driving record revenue
Thesis: Leading AI semiconductor company with dominant GPU market position

**STOCK 2:**
Symbol: PLTR
Company: Palantir Technologies Inc
Founded: 2003
Performance: 12.8
Category: mid-cap
Catalyst: Government contracts expansion and AI platform adoption
Thesis: Data analytics leader with strong government and enterprise growth

**STOCK 3:**
Symbol: SNOW
Company: Snowflake Inc
Founded: 2012
Performance: 8.5
Category: mid-cap
Catalyst: Cloud data platform expansion and AI integration
Thesis: Fast-growing cloud data warehouse with expanding market share"""

🤖 Calling OpenAI API...
✅ AI response received!
   Response length: 3537 characters
   Model used: gpt-4o-mini-2024-07-18

📝 Response preview:
Here are 10 trending stocks in the Technology sector for July 2025, featuring a mix of established leaders and emerging growth companies:

**STOCK 1:**
Symbol: AAPL  
Company: Apple Inc.  
Founded: 1976  
Performance: 15%  
Category: large-cap  
Catalyst: Strong demand for the upcoming iPhone releas...


In [10]:
ai_response

"Here are 10 trending stocks in the Technology sector for July 2025, featuring a mix of established leaders and emerging growth companies:\n\n**STOCK 1:**\nSymbol: AAPL  \nCompany: Apple Inc.  \nFounded: 1976  \nPerformance: 15%  \nCategory: large-cap  \nCatalyst: Strong demand for the upcoming iPhone release and expansion into AR/VR technologies.  \nThesis: Apple continues to innovate and diversify its product line, which positions it well for sustained growth.\n\n**STOCK 2:**\nSymbol: MSFT  \nCompany: Microsoft Corporation  \nFounded: 1975  \nPerformance: 12%  \nCategory: large-cap  \nCatalyst: Successful integration of AI capabilities into existing products like Office and Azure.  \nThesis: Microsoft's commitment to AI and cloud services drives revenue growth and market leadership.\n\n**STOCK 3:**\nSymbol: SNOW  \nCompany: Snowflake Inc.  \nFounded: 2012  \nPerformance: 20%  \nCategory: mid-cap  \nCatalyst: Increasing demand for cloud data warehousing solutions and partnerships with

### 9. PARSE AI RESPONSE

In [9]:
print("🔍 Parsing AI response to extract stock data...")

ai_stocks = []

if ai_response:
    # Split response into individual stock blocks
    stock_blocks = re.split(r'\*\*STOCK \d+:\*\*', ai_response)
    
    print(f"   Found {len(stock_blocks)-1} stock blocks in response")
    
    # Process each stock block (skip first empty block)
    for i, block in enumerate(stock_blocks[1:], 1):
        stock_data = {}
        
        print(f"\n   Processing Stock {i}:")
        
        # Extract symbol
        symbol_match = re.search(r'Symbol:\s*\[?([A-Z]{1,5})\]?', block, re.IGNORECASE)
        if symbol_match:
            stock_data['symbol'] = symbol_match.group(1).strip()
            print(f"      Symbol: {stock_data['symbol']}")
        
        # Extract company name
        company_match = re.search(r'Company:\s*([^\n]+)', block, re.IGNORECASE)
        if company_match:
            stock_data['company_name'] = company_match.group(1).strip()
            print(f"      Company: {stock_data['company_name']}")
        
        # Extract founded year
        year_match = re.search(r'Founded:\s*(\d{4})', block, re.IGNORECASE)
        if year_match:
            stock_data['founded_year'] = year_match.group(1)
            print(f"      Founded: {stock_data['founded_year']}")
        
        # Extract expected performance
        perf_match = re.search(r'Performance:\s*\[?([\d.-]+)%?\]?', block, re.IGNORECASE)
        if perf_match:
            try:
                stock_data['ai_expected_performance'] = float(perf_match.group(1))
                print(f"      Expected: {stock_data['ai_expected_performance']}%")
            except:
                stock_data['ai_expected_performance'] = random.uniform(5.0, 15.0)
        
        # Extract category
        cat_match = re.search(r'Category:\s*([^\n]+)', block, re.IGNORECASE)
        if cat_match:
            stock_data['market_cap_category'] = cat_match.group(1).strip()
            print(f"      Category: {stock_data['market_cap_category']}")
        
        # Extract catalyst
        catalyst_match = re.search(r'Catalyst:\s*([^\n]+)', block, re.IGNORECASE)
        if catalyst_match:
            stock_data['ai_catalyst'] = catalyst_match.group(1).strip()
            print(f"      Catalyst: {stock_data['ai_catalyst'][:50]}...")
        
        # Extract thesis
        thesis_match = re.search(r'Thesis:\s*([^\n]+)', block, re.IGNORECASE)
        if thesis_match:
            stock_data['ai_thesis'] = thesis_match.group(1).strip()
            print(f"      Thesis: {stock_data['ai_thesis'][:50]}...")
        
        # Validate and add to list
        if stock_data.get('symbol') and len(stock_data.get('symbol', '')) <= 5:
            ai_stocks.append(stock_data)
            print(f"      ✅ Added {stock_data['symbol']}")
        else:
            print(f"      ❌ Invalid or missing symbol")

print(f"\n📊 Successfully parsed {len(ai_stocks)} stocks from AI response")

🔍 Parsing AI response to extract stock data...
   Found 10 stock blocks in response

   Processing Stock 1:
      Symbol: AAPL
      Company: Apple Inc.
      Founded: 1976
      Expected: 15.0%
      Category: large-cap
      Catalyst: Strong demand for the upcoming iPhone release and ...
      Thesis: Apple continues to innovate and diversify its prod...
      ✅ Added AAPL

   Processing Stock 2:
      Symbol: MSFT
      Company: Microsoft Corporation
      Founded: 1975
      Expected: 12.0%
      Category: large-cap
      Catalyst: Successful integration of AI capabilities into exi...
      Thesis: Microsoft's commitment to AI and cloud services dr...
      ✅ Added MSFT

   Processing Stock 3:
      Symbol: SNOW
      Company: Snowflake Inc.
      Founded: 2012
      Expected: 20.0%
      Category: mid-cap
      Catalyst: Increasing demand for cloud data warehousing solut...
      Thesis: As businesses increasingly move to cloud solutions...
      ✅ Added SNOW

   Processing Stock 

### 10. DISPLAY AI DISCOVERED STOCKS

In [11]:
print("🤖 AI DISCOVERED STOCKS:")
print("=" * 80)

for i, stock in enumerate(ai_stocks, 1):
    print(f"Stock {i}: {stock.get('symbol', 'N/A')}")
    print(f"   Company: {stock.get('company_name', 'N/A')}")
    print(f"   Founded: {stock.get('founded_year', 'N/A')}")
    print(f"   AI Expected Return: {stock.get('ai_expected_performance', 0)}%")
    print(f"   Category: {stock.get('market_cap_category', 'N/A')}")
    print(f"   Catalyst: {stock.get('ai_catalyst', 'N/A')[:60]}...")
    print()

print("=" * 80)
print(f"Total AI discovered stocks: {len(ai_stocks)}")

🤖 AI DISCOVERED STOCKS:
Stock 1: AAPL
   Company: Apple Inc.
   Founded: 1976
   AI Expected Return: 15.0%
   Category: large-cap
   Catalyst: Strong demand for the upcoming iPhone release and expansion ...

Stock 2: MSFT
   Company: Microsoft Corporation
   Founded: 1975
   AI Expected Return: 12.0%
   Category: large-cap
   Catalyst: Successful integration of AI capabilities into existing prod...

Stock 3: SNOW
   Company: Snowflake Inc.
   Founded: 2012
   AI Expected Return: 20.0%
   Category: mid-cap
   Catalyst: Increasing demand for cloud data warehousing solutions and p...

Stock 4: ZS
   Company: Zscaler, Inc.
   Founded: 2008
   AI Expected Return: 18.0%
   Category: mid-cap
   Catalyst: Rising cybersecurity threats leading to increased investment...

Stock 5: CRWD
   Company: CrowdStrike Holdings, Inc.
   Founded: 2011
   AI Expected Return: 22.0%
   Category: mid-cap
   Catalyst: Recent acquisition of a key cybersecurity firm to enhance it...

Stock 6: PLTR
   Company: Pala

### 11. GET MARKET DATA FOR AI STOCKS

In [13]:



print("📡 Fetching market data for AI-discovered stocks...")

ai_symbols = [stock.get('symbol', '') for stock in ai_stocks if stock.get('symbol')]
ai_market_data = {}

print(f"   AI symbols to fetch: {ai_symbols}")

for symbol in ai_symbols:
    if symbol in market_data:
        # We already have this data
        ai_market_data[symbol] = market_data[symbol]
        print(f"   ✅ {symbol}: Using existing data")
    else:
        # Fetch new data
        try:
            print(f"   📡 Fetching new data for {symbol}...")
            ticker = yf.Ticker(symbol)
            hist_1m = ticker.history(period="1mo")
            info = ticker.info
            
            if not hist_1m.empty:
                current_price = hist_1m['Close'].iloc[-1]
                start_1m = hist_1m['Close'].iloc[0]
                performance_1m = ((current_price - start_1m) / start_1m) * 100
                
                ai_market_data[symbol] = {
                    "current_price": round(float(current_price), 2),
                    "1m_performance": round(float(performance_1m), 2),
                    "sector": info.get("sector", "Unknown"),
                    "market_cap": info.get("marketCap", 0),
                    "company_name": info.get("longName", symbol)
                }
                print(f"      ✅ ${current_price:.2f}, 1M: {performance_1m:+.1f}%")
            else:
                print(f"      ❌ No data available")
                
        except Exception as e:
            print(f"      ❌ Error: {e}")
            continue

print(f"\n📊 Market data available for {len(ai_market_data)} AI stocks")

📡 Fetching market data for AI-discovered stocks...
   AI symbols to fetch: ['AAPL', 'MSFT', 'SNOW', 'ZS', 'CRWD', 'PLTR', 'RBLX', 'AI', 'DOCU', 'FSLY']
   ✅ AAPL: Using existing data
   ✅ MSFT: Using existing data
   📡 Fetching new data for SNOW...
      ✅ $219.26, 1M: -2.0%
   📡 Fetching new data for ZS...
      ✅ $288.99, 1M: -7.9%
   📡 Fetching new data for CRWD...
      ✅ $471.51, 1M: -7.4%
   📡 Fetching new data for PLTR...
      ✅ $156.21, 1M: +14.6%
   📡 Fetching new data for RBLX...
      ✅ $119.63, 1M: +13.7%
   📡 Fetching new data for AI...
      ✅ $26.02, 1M: +5.9%
   📡 Fetching new data for DOCU...
      ✅ $80.90, 1M: +3.9%
   📡 Fetching new data for FSLY...
      ✅ $7.36, 1M: +4.3%

📊 Market data available for 10 AI stocks


### 12. COMBINE AI DATA WITH MARKET DATA

In [14]:
print("🔄 Combining AI insights with real market data...")

enriched_stocks = []

for stock_info in ai_stocks:
    symbol = stock_info.get('symbol', '')
    
    if symbol in ai_market_data:
        # Combine AI data with market data
        combined_data = ai_market_data[symbol].copy()  # Start with market data
        
        # Add AI insights
        combined_data.update({
            'ai_company_name': stock_info.get('company_name', symbol),
            'ai_founded_year': stock_info.get('founded_year', 'N/A'),
            'ai_catalyst': stock_info.get('ai_catalyst', 'Market trends'),
            'ai_thesis': stock_info.get('ai_thesis', 'AI-identified opportunity'),
            'ai_expected_performance': stock_info.get('ai_expected_performance', 0),
            'market_cap_category': stock_info.get('market_cap_category', 'Unknown')
        })
        
        enriched_stocks.append((symbol, combined_data))
        print(f"   ✅ {symbol}: Combined AI + Market data")
        
        # Show the combination
        ai_expected = combined_data.get('ai_expected_performance', 0)
        real_1m = combined_data.get('1m_performance', 0)
        print(f"      AI Prediction: {ai_expected:+.1f}% vs Real 1M: {real_1m:+.1f}%")
        
    else:
        print(f"   ⚠️ {symbol}: No market data available")

print(f"\n📊 Successfully enriched {len(enriched_stocks)} stocks")

🔄 Combining AI insights with real market data...
   ✅ AAPL: Combined AI + Market data
      AI Prediction: +15.0% vs Real 1M: +4.2%
   ✅ MSFT: Combined AI + Market data
      AI Prediction: +12.0% vs Real 1M: +2.9%
   ✅ SNOW: Combined AI + Market data
      AI Prediction: +20.0% vs Real 1M: -2.0%
   ✅ ZS: Combined AI + Market data
      AI Prediction: +18.0% vs Real 1M: -8.0%
   ✅ CRWD: Combined AI + Market data
      AI Prediction: +22.0% vs Real 1M: -7.4%
   ✅ PLTR: Combined AI + Market data
      AI Prediction: +25.0% vs Real 1M: +14.6%
   ✅ RBLX: Combined AI + Market data
      AI Prediction: +30.0% vs Real 1M: +13.7%
   ✅ AI: Combined AI + Market data
      AI Prediction: +28.0% vs Real 1M: +5.9%
   ✅ DOCU: Combined AI + Market data
      AI Prediction: +10.0% vs Real 1M: +3.9%
   ✅ FSLY: Combined AI + Market data
      AI Prediction: +15.0% vs Real 1M: +4.3%

📊 Successfully enriched 10 stocks


### 13. SORT BY PERFORMANCE

In [15]:
print("📊 Sorting stocks by 1-month performance...")

# Sort enriched stocks by 1-month performance (descending)
enriched_stocks.sort(key=lambda x: x[1].get('1m_performance', 0), reverse=True)

print("\n🏆 PERFORMANCE RANKING:")
print("=" * 80)

for i, (symbol, data) in enumerate(enriched_stocks, 1):
    performance = data.get('1m_performance', 0)
    price = data.get('current_price', 0)
    company = data.get('ai_company_name', symbol)
    
    print(f"{i:2}. {symbol:6} | ${price:8.2f} | {performance:+6.1f}% | {company}")

print("=" * 80)

📊 Sorting stocks by 1-month performance...

🏆 PERFORMANCE RANKING:
 1. PLTR   | $  156.21 |  +14.6% | Palantir Technologies Inc.
 2. RBLX   | $  119.63 |  +13.7% | Roblox Corporation
 3. AI     | $   26.02 |   +5.9% | C3.ai, Inc.
 4. FSLY   | $    7.36 |   +4.3% | Fastly, Inc.
 5. AAPL   | $  213.82 |   +4.2% | Apple Inc.
 6. DOCU   | $   80.90 |   +3.9% | DocuSign, Inc.
 7. MSFT   | $  511.71 |   +2.9% | Microsoft Corporation
 8. SNOW   | $  219.26 |   -2.0% | Snowflake Inc.
 9. CRWD   | $  471.51 |   -7.4% | CrowdStrike Holdings, Inc.
10. ZS     | $  288.99 |   -8.0% | Zscaler, Inc.


### 14. CALCULATE RATING THRESHOLDS

In [16]:
print("📊 Calculating rating thresholds based on performance...")

# Extract all 1-month performances
performances = [data.get('1m_performance', 0) for _, data in enriched_stocks]

print(f"   Performance data points: {len(performances)}")
print(f"   Best performance: {max(performances):+.1f}%")
print(f"   Worst performance: {min(performances):+.1f}%")

if len(performances) >= 4:
    # Sort performances to calculate percentiles
    sorted_perfs = sorted(performances, reverse=True)
    n = len(sorted_perfs)
    
    # Calculate dynamic thresholds
    strong_buy_threshold = sorted_perfs[min(n//4, len(sorted_perfs)-1)]  # Top 25%
    moderate_buy_threshold = sorted_perfs[min(n//2, len(sorted_perfs)-1)]  # Top 50%
    hold_threshold = sorted_perfs[min(3*n//4, len(sorted_perfs)-1)]  # Top 75%
    # Bottom 25% = Sell
    
else:
    # Fallback thresholds for small datasets
    strong_buy_threshold = 5.0
    moderate_buy_threshold = 0.0
    hold_threshold = -5.0

print(f"\n🎯 RATING THRESHOLDS:")
print(f"   🟢 Strong Buy: > {strong_buy_threshold:+.1f}%")
print(f"   🟡 Moderate Buy: > {moderate_buy_threshold:+.1f}%")
print(f"   🔵 Hold: > {hold_threshold:+.1f}%")
print(f"   🔴 Sell: ≤ {hold_threshold:+.1f}%")

📊 Calculating rating thresholds based on performance...
   Performance data points: 10
   Best performance: +14.6%
   Worst performance: -8.0%

🎯 RATING THRESHOLDS:
   🟢 Strong Buy: > +5.9%
   🟡 Moderate Buy: > +3.9%
   🔵 Hold: > -2.0%
   🔴 Sell: ≤ -2.0%


### 15. ASSIGN RATINGS

In [17]:
print("🏷️ Assigning ratings to each stock...")

# Initialize categories
rating_categories = {
    "strong_buy": [],
    "moderate_buy": [],
    "hold": [],
    "sell": []
}

# Assign ratings based on performance
for symbol, data in enriched_stocks:
    performance = data.get('1m_performance', 0)
    
    if performance >= strong_buy_threshold:
        rating = "🟢 Strong Buy"
        category = "strong_buy"
    elif performance >= moderate_buy_threshold:
        rating = "🟡 Moderate Buy"
        category = "moderate_buy"
    elif performance >= hold_threshold:
        rating = "🔵 Hold"
        category = "hold"
    else:
        rating = "🔴 Sell"
        category = "sell"
    
    # Add rating to data
    data['rating'] = rating
    data['rating_category'] = category
    
    # Add to category list
    rating_categories[category].append((symbol, data))
    
    print(f"   {symbol:6} | {performance:+6.1f}% → {rating}")

print(f"\n📊 RATING DISTRIBUTION:")
for category, stocks in rating_categories.items():
    category_name = category.replace('_', ' ').title()
    print(f"   {category_name}: {len(stocks)} stocks")

🏷️ Assigning ratings to each stock...
   PLTR   |  +14.6% → 🟢 Strong Buy
   RBLX   |  +13.7% → 🟢 Strong Buy
   AI     |   +5.9% → 🟢 Strong Buy
   FSLY   |   +4.3% → 🟡 Moderate Buy
   AAPL   |   +4.2% → 🟡 Moderate Buy
   DOCU   |   +3.9% → 🟡 Moderate Buy
   MSFT   |   +2.9% → 🔵 Hold
   SNOW   |   -2.0% → 🔵 Hold
   CRWD   |   -7.4% → 🔴 Sell
   ZS     |   -8.0% → 🔴 Sell

📊 RATING DISTRIBUTION:
   Strong Buy: 3 stocks
   Moderate Buy: 3 stocks
   Hold: 2 stocks
   Sell: 2 stocks


### 16. CREATE REPORT HEADER

In [18]:
print("📝 Creating trending assets report...")

current_date_formatted = datetime.now().strftime('%B %d, %Y')
total_stocks = len(enriched_stocks)

report_header = f"""# Trending Stocks Analysis Report – {current_date_formatted}

## Executive Summary

**Analysis Focus**: Trending stocks in {sector} sector  
**Market Outlook**: Based on 1-month performance analysis  
**Research Method**: AI agent discovery + real-time data validation  
**Total Assets Analyzed**: {total_stocks}
**Data Source**: Yahoo Finance API + OpenAI GPT-4o-mini

---

"""

print("✅ Report header created")
print(f"   Date: {current_date_formatted}")
print(f"   Total stocks: {total_stocks}")

📝 Creating trending assets report...
✅ Report header created
   Date: July 28, 2025
   Total stocks: 10


### 17. GENERATE RATING SECTIONS

In [19]:
print("📊 Generating rating sections for report...")

rating_sections = ""

# Process each rating category
for category, stocks in rating_categories.items():
    if not stocks:
        continue
    
    category_title = category.replace('_', ' ').title()
    category_emoji = {
        "strong_buy": "🟢",
        "moderate_buy": "🟡", 
        "hold": "🔵",
        "sell": "🔴"
    }.get(category, "")
    
    print(f"   Creating {category_title} section ({len(stocks)} stocks)")
    
    rating_sections += f"""## {category_emoji} {category_title} Recommendations

| Symbol | Company Name | Current Price | 1M Return | Market Cap | AI Catalyst |
|--------|--------------|---------------|-----------|------------|-------------|
"""
    
    # Add up to 4 stocks per category
    for symbol, data in stocks[:4]:
        price = data.get('current_price', 0)
        performance = data.get('1m_performance', 0)
        
        # Format market cap
        market_cap = data.get('market_cap', 0)
        if market_cap > 1e12:
            cap_display = f"${market_cap/1e12:.1f}T"
        elif market_cap > 1e9:
            cap_display = f"${market_cap/1e9:.1f}B"
        else:
            cap_display = f"${market_cap/1e6:.0f}M"
        
        company_name = data.get('ai_company_name', symbol)
        catalyst = data.get('ai_catalyst', 'Market trends')
        
        # Truncate catalyst if too long
        if len(catalyst) > 50:
            catalyst = catalyst[:50] + "..."
        
        rating_sections += f"| {symbol} | {company_name} | ${price:.2f} | {performance:+.1f}% | {cap_display} | {catalyst} |\n"
    
    rating_sections += "\n"

print("✅ Rating sections created")

📊 Generating rating sections for report...
   Creating Strong Buy section (3 stocks)
   Creating Moderate Buy section (3 stocks)
   Creating Hold section (2 stocks)
   Creating Sell section (2 stocks)
✅ Rating sections created


### 18. CREATE INVESTMENT STRATEGY SECTION

In [20]:

print("📋 Creating investment strategy section...")

strategy_section = f"""## Investment Strategy for Trending Stocks

### Performance-Based Approach

The ratings above are based on **dynamic performance percentiles** rather than fixed thresholds, ensuring balanced recommendations:

- **Strong Buy ({len(rating_categories['strong_buy'])} stocks)**: Top 25% performers with positive momentum
- **Moderate Buy ({len(rating_categories['moderate_buy'])} stocks)**: Top 50% performers with stable trends  
- **Hold ({len(rating_categories['hold'])} stocks)**: Top 75% performers requiring monitoring
- **Sell ({len(rating_categories['sell'])} stocks)**: Bottom 25% performers showing weakness

### Risk Management Guidelines

1. **Position Sizing**: Limit individual positions to 3-5% of portfolio
2. **Diversification**: Spread across multiple sectors and market caps
3. **Regular Review**: Monthly rebalancing based on performance updates
4. **Stop Losses**: Consider 15-20% stop losses for volatile positions

### Key Sector Insights

**{sector} Sector Trends**:
- AI and machine learning continue to drive growth
- Cloud computing adoption accelerating
- Hardware demand supporting semiconductor stocks
- Regulatory environment remains supportive

---

"""

print("✅ Investment strategy section created")

📋 Creating investment strategy section...
✅ Investment strategy section created


### 19. CREATE METHODOLOGY SECTION

In [21]:
print("📖 Creating methodology section...")

methodology_section = f"""## Methodology & Data Sources

### AI Discovery Process
- **Model**: OpenAI GPT-4o-mini with temperature {temperature:.2f}
- **Context**: {market_context}
- **Session**: {session_id}
- **Discovered Stocks**: {len(ai_stocks)} initial recommendations

### Market Data Integration
- **Source**: Yahoo Finance API
- **Metrics**: Real-time prices, 1-month performance, market cap, sector
- **Validation**: {len(ai_market_data)} stocks with confirmed data

### Rating Methodology
- **Dynamic Thresholds**: Performance percentile-based classification
- **Strong Buy**: Top 25% (> {strong_buy_threshold:+.1f}%)
- **Moderate Buy**: Top 50% (> {moderate_buy_threshold:+.1f}%)
- **Hold**: Top 75% (> {hold_threshold:+.1f}%)
- **Sell**: Bottom 25%

### Update Frequency
- **Real-time data**: Live market prices and performance
- **AI insights**: Dynamic discovery with varied results
- **Report generation**: On-demand analysis

---

## Investment Disclaimer

*This analysis is for educational and informational purposes only. All investments carry inherent risks, and past performance does not guarantee future results. The AI-generated recommendations should not be considered as personalized investment advice. Always conduct your own research and consult with qualified financial professionals before making investment decisions.*

---

**Report Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} by AI Investment Advisor  
**Powered by**: OpenAI GPT-4o-mini + Yahoo Finance API
"""

print("✅ Methodology section created")


📖 Creating methodology section...
✅ Methodology section created


### 20. COMBINE FULL REPORT

In [22]:
print("📄 Combining all sections into final report...")

# Combine all sections
full_report = report_header + rating_sections + strategy_section + methodology_section

# Report statistics
report_length = len(full_report)
word_count = len(full_report.split())
line_count = full_report.count('\n')

print(f"✅ Full report created!")
print(f"   📊 Length: {report_length:,} characters")
print(f"   📝 Words: {word_count:,}")
print(f"   📄 Lines: {line_count}")

📄 Combining all sections into final report...
✅ Full report created!
   📊 Length: 4,655 characters
   📝 Words: 676
   📄 Lines: 105


### 21. DISPLAY REPORT PREVIEW

In [23]:
print("👀 REPORT PREVIEW (First 1000 characters):")
print("=" * 80)
print(full_report[:1000])
print("...")
print("=" * 80)

print(f"\n📄 Complete report is {len(full_report)} characters long")
print("   (Scroll up to see the full content)")

👀 REPORT PREVIEW (First 1000 characters):
# Trending Stocks Analysis Report – July 28, 2025

## Executive Summary

**Analysis Focus**: Trending stocks in Technology sector  
**Market Outlook**: Based on 1-month performance analysis  
**Research Method**: AI agent discovery + real-time data validation  
**Total Assets Analyzed**: 10
**Data Source**: Yahoo Finance API + OpenAI GPT-4o-mini

---

## 🟢 Strong Buy Recommendations

| Symbol | Company Name | Current Price | 1M Return | Market Cap | AI Catalyst |
|--------|--------------|---------------|-----------|------------|-------------|
| PLTR | Palantir Technologies Inc. | $156.21 | +14.6% | $368.7B | New government contracts and expansion into commer... |
| RBLX | Roblox Corporation | $119.63 | +13.7% | $81.1B | Expansion of its metaverse platform and partnershi... |
| AI | C3.ai, Inc. | $26.02 | +5.9% | $3.5B | Increased adoption of AI solutions across various ... |

## 🟡 Moderate Buy Recommendations

| Symbol | Company Name | Current 

### 22. CREATE SUMMARY DATAFRAMES

In [24]:
print("📊 Creating summary DataFrames for analysis...")

# Create trending assets DataFrame (Strong Buy + Moderate Buy)
trending_data = []
stable_data = []

for category, stocks in rating_categories.items():
    for symbol, data in stocks:
        # Format market cap
        market_cap = data.get('market_cap', 0)
        if market_cap > 1e12:
            cap_display = f"${market_cap/1e12:.1f}T"
        elif market_cap > 1e9:
            cap_display = f"${market_cap/1e9:.1f}B"
        else:
            cap_display = f"${market_cap/1e6:.0f}M"
        
        row = {
            "Symbol": symbol,
            "Company": data.get('ai_company_name', symbol),
            "Price": f"${data.get('current_price', 0):.2f}",
            "1M Return": f"{data.get('1m_performance', 0):+.1f}%",
            "Rating": data.get('rating', 'N/A'),
            "Market Cap": cap_display,
            "Sector": data.get('sector', 'Unknown'),
            "AI Expected": f"{data.get('ai_expected_performance', 0):+.1f}%"
        }
        
        if category in ['strong_buy', 'moderate_buy']:
            trending_data.append(row)
        else:
            stable_data.append(row)

# Create DataFrames
trending_df = pd.DataFrame(trending_data)
stable_df = pd.DataFrame(stable_data)

print(f"✅ DataFrames created!")
print(f"   📈 Trending assets: {len(trending_df)} rows")
print(f"   📊 Stable assets: {len(stable_df)} rows")

📊 Creating summary DataFrames for analysis...
✅ DataFrames created!
   📈 Trending assets: 6 rows
   📊 Stable assets: 4 rows


### 23. DISPLAY SUMMARY TABLES

In [25]:
print("📈 TRENDING ASSETS SUMMARY (Strong Buy + Moderate Buy):")
print("=" * 100)
if not trending_df.empty:
    print(trending_df.to_string(index=False))
else:
    print("No trending assets found")
print("=" * 100)

print(f"\n📊 STABLE ASSETS SUMMARY (Hold + Sell):")
print("=" * 100)
if not stable_df.empty:
    print(stable_df.to_string(index=False))
else:
    print("No stable assets found")
print("=" * 100)

📈 TRENDING ASSETS SUMMARY (Strong Buy + Moderate Buy):
Symbol                    Company   Price 1M Return         Rating Market Cap                 Sector AI Expected
  PLTR Palantir Technologies Inc. $156.21    +14.6%   🟢 Strong Buy    $368.7B             Technology      +25.0%
  RBLX         Roblox Corporation $119.63    +13.7%   🟢 Strong Buy     $81.1B Communication Services      +30.0%
    AI                C3.ai, Inc.  $26.02     +5.9%   🟢 Strong Buy      $3.5B             Technology      +28.0%
  FSLY               Fastly, Inc.   $7.36     +4.3% 🟡 Moderate Buy      $1.1B             Technology      +15.0%
  AAPL                 Apple Inc. $213.82     +4.2% 🟡 Moderate Buy      $3.2T             Technology      +15.0%
  DOCU             DocuSign, Inc.  $80.90     +3.9% 🟡 Moderate Buy     $16.3B             Technology      +10.0%

📊 STABLE ASSETS SUMMARY (Hold + Sell):
Symbol                    Company   Price 1M Return Rating Market Cap     Sector AI Expected
  MSFT      Microsoft

### 24. FINAL SUMMARY & STATISTICS

In [26]:
print("🎯 FINAL ANALYSIS SUMMARY:")
print("=" * 60)

print(f"📊 DATA PROCESSING:")
print(f"   • AI discovered stocks: {len(ai_stocks)}")
print(f"   • Market data fetched: {len(ai_market_data)}")
print(f"   • Successfully enriched: {len(enriched_stocks)}")

print(f"\n🏷️ RATING DISTRIBUTION:")
for category, stocks in rating_categories.items():
    emoji = {"strong_buy": "🟢", "moderate_buy": "🟡", "hold": "🔵", "sell": "🔴"}.get(category, "")
    print(f"   • {emoji} {category.replace('_', ' ').title()}: {len(stocks)} stocks")

print(f"\n📄 REPORT STATISTICS:")
print(f"   • Report length: {len(full_report):,} characters")
print(f"   • Word count: {len(full_report.split()):,}")
print(f"   • Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n📈 PERFORMANCE RANGE:")
if performances:
    print(f"   • Best performer: {max(performances):+.1f}%")
    print(f"   • Worst performer: {min(performances):+.1f}%")
    print(f"   • Average performance: {sum(performances)/len(performances):+.1f}%")

print("=" * 60)
print("✅ Analysis complete! All data is ready for use.")

🎯 FINAL ANALYSIS SUMMARY:
📊 DATA PROCESSING:
   • AI discovered stocks: 10
   • Market data fetched: 10
   • Successfully enriched: 10

🏷️ RATING DISTRIBUTION:
   • 🟢 Strong Buy: 3 stocks
   • 🟡 Moderate Buy: 3 stocks
   • 🔵 Hold: 2 stocks
   • 🔴 Sell: 2 stocks

📄 REPORT STATISTICS:
   • Report length: 4,655 characters
   • Word count: 676
   • Generated: 2025-07-28 12:20:14

📈 PERFORMANCE RANGE:
   • Best performer: +14.6%
   • Worst performer: -8.0%
   • Average performance: +3.2%
✅ Analysis complete! All data is ready for use.


### 25. EXPORT OPTIONS (OPTIONAL)

In [27]:
print("💾 EXPORT OPTIONS:")
print("   To save the report to file, run:")
print("   with open('trending_stocks_report.md', 'w') as f:")
print("       f.write(full_report)")
print()
print("   To save DataFrames to CSV, run:")
print("   trending_df.to_csv('trending_stocks.csv', index=False)")
print("   stable_df.to_csv('stable_stocks.csv', index=False)")
print()
print("🔄 To run analysis again with different results:")
print("   • Restart from cell 6 (AI Discovery)")
print("   • The randomization will give different AI recommendations")
print()
print("🎉 Notebook execution complete!")

💾 EXPORT OPTIONS:
   To save the report to file, run:
   with open('trending_stocks_report.md', 'w') as f:
       f.write(full_report)

   To save DataFrames to CSV, run:
   trending_df.to_csv('trending_stocks.csv', index=False)
   stable_df.to_csv('stable_stocks.csv', index=False)

🔄 To run analysis again with different results:
   • Restart from cell 6 (AI Discovery)
   • The randomization will give different AI recommendations

🎉 Notebook execution complete!
